<a href="https://colab.research.google.com/github/labonig/CMSC-487-HW-4/blob/main/_downloads/4e865243430a47a00d551ca0579a6f6c/cifar10_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The code here is modified from the cifar10_tutorial.

In [2]:
%matplotlib inline

### 1. Load and normalize CIFAR10


In [3]:
import torch
import torchvision
import torchvision.transforms as transforms

In [4]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [02:50<00:00, 998kB/s] 


2. Define a Convolutional Neural Network
========================================


In [24]:
# Question 0: Basic CNN
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()

In [12]:
# Question 1: Deeper CNN architecture
class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()

        #convolutional block 1
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.conv2 = nn.Conv2d(6, 12, 5)

        self.pool = nn.MaxPool2d(2, 2)

        #convolutional block 2
        self.conv3 = nn.Conv2d(12, 24, 5)
        self.conv4 = nn.Conv2d(24, 48, 5)

        self.fc1 = nn.Linear(48 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # first convolutional layer
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)

        # second convolutional layer
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)

        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

deeper_net = DeepNet()
print(deeper_net)

DeepNet(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 12, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(12, 24, kernel_size=(5, 5), stride=(1, 1))
  (conv4): Conv2d(24, 48, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=1200, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


3. Define a Loss function and optimizer
=======================================


In [25]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

4. Train the network
====================

In [26]:
def train_network(nn_name, num_epochs):
  for epoch in range(num_epochs):  # loop over the dataset multiple times

      running_loss = 0.0
      for i, data in enumerate(trainloader, 0):
          # get the inputs; data is a list of [inputs, labels]
          inputs, labels = data

          # zero the parameter gradients
          optimizer.zero_grad()

          # forward + backward + optimize
          outputs = nn_name(inputs)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          # print statistics
          running_loss += loss.item()
          if i % 2000 == 1999:    # print every 2000 mini-batches
              print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
              running_loss = 0.0

  print('Finished Training')


In [27]:
# Training the tutorial network
train_network(net, 2)

[1,  2000] loss: 2.139
[1,  4000] loss: 1.841
[1,  6000] loss: 1.680
[1,  8000] loss: 1.569
[1, 10000] loss: 1.510
[1, 12000] loss: 1.471
[2,  2000] loss: 1.396
[2,  4000] loss: 1.381
[2,  6000] loss: 1.319
[2,  8000] loss: 1.325
[2, 10000] loss: 1.296
[2, 12000] loss: 1.253
Finished Training


In [28]:
# save the trained model:
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

In [13]:
# Training the deeper model with convolutional blocks
train_network(deeper_net, 10)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x192 and 1200x120)

In [ ]:
# save the trained model:
PATH2 = './cifar_deeper_net.pth'
torch.save(deeper_net.state_dict(), PATH2)

5. Test the network on the test data
====================================

In [29]:
dataiter = iter(testloader)
images, labels = next(dataiter)

In [30]:
# loading back the saved model:
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))

<All keys matched successfully>

In [31]:
def test_network(nn_name):
  correct = 0
  total = 0
  # since we're not training, we don't need to calculate the gradients for our outputs
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          # calculate outputs by running images through the network
          outputs = nn_name(images)
          # the class with the highest energy is what we choose as prediction
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

In [32]:
# Testing the tutorial network
test_network(net)

Accuracy of the network on the 10000 test images: 55 %


In [33]:
def print_accuracy(nn_name):
  # prepare to count predictions for each class
  correct_pred = {classname: 0 for classname in classes}
  total_pred = {classname: 0 for classname in classes}

  # again no gradients needed
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          outputs = nn_name(images)
          _, predictions = torch.max(outputs, 1)
          # collect the correct predictions for each class
          for label, prediction in zip(labels, predictions):
              if label == prediction:
                  correct_pred[classes[label]] += 1
              total_pred[classes[label]] += 1

  # print accuracy for each class
  for classname, correct_count in correct_pred.items():
      accuracy = 100 * float(correct_count) / total_pred[classname]
      print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

In [34]:
# print the per-class classification accuracy of the tutorial network
print_accuracy(net)

Accuracy for class: plane is 58.4 %
Accuracy for class: car   is 76.5 %
Accuracy for class: bird  is 38.7 %
Accuracy for class: cat   is 36.7 %
Accuracy for class: deer  is 40.4 %
Accuracy for class: dog   is 42.4 %
Accuracy for class: frog  is 85.3 %
Accuracy for class: horse is 59.2 %
Accuracy for class: ship  is 65.1 %
Accuracy for class: truck is 50.2 %


Training on GPU
===============

Just like how you transfer a Tensor onto the GPU, you transfer the
neural net onto the GPU.

Let\'s first define our device as the first visible cuda device if we
have CUDA available:


In [9]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

cuda:0


The rest of this section assumes that `device` is a CUDA device.

Then these methods will recursively go over all modules and convert
their parameters and buffers to CUDA tensors:

``` {.python}
net.to(device)
```

Remember that you will have to send the inputs and targets at every step
to the GPU too:

``` {.python}
inputs, labels = data[0].to(device), data[1].to(device)
```

In [16]:
del dataiter